# Final submission pipeline

Self-contained notebook: place `X_test` in the `data/` directory, run all cells, and get ready-to-submit files.

---

## What this notebook does

1. Loads training data (`X_train`, `y_train`) and test data (`X_test`)
2. Trains the best model on the **full** `X_train` (5 000 observations)
3. Scores all 5 000 `X_test` observations
4. Selects the **top-997** observations by predicted score (k optimised on pooled OOF)
5. Writes two submission files:
   - `_obs.txt` — indices of the selected observations (rows of X_test, 1-indexed)
   - `_vars.txt` — indices of the selected features (1-indexed, V1=1)

---

## Best model (selected by nested CV)

| Parameter | Value |
|---|---|
| **Algorithm** | ExtraTreesClassifier |
| **Prescreening** | sparse GAM SPAM (ps_mean_0007) |
| **Final features** | `var_175, var_190, var_214, var_341, var_379, var_482` (6 features) |
| **class_weight** | balanced |
| **max_depth** | 5 |
| **min_samples_leaf** | 16 |
| **n_estimators** | 400 |
| **OOF business score** | 5 425 (pooled 5×1000, nested-CV OOF estimate) |
| **In-sample business score** | ~5 790 (in-sample upper bound) |
| **Optimal k (OOF)** | 997 (~20% contact rate) |

---

## Business score formula (objective)

$$\text{score} = 10 \cdot TP - 5 \cdot FP - 200 \cdot |\text{features}|$$

- **TP** (True Positive): selected observation that is a genuine customer
- **FP** (False Positive): selected observation that is NOT a customer
- **200 × 6 = 1 200**: fixed cost for using 6 features

> **k = 997** means we contact 997 out of 5 000 test observations (≈ 19.94% contact rate).
> The value was determined as argmax business score on pooled OOF (5×1000 disjoint predictions).

## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import ExtraTreesClassifier

from cost_effective.dataset import find_project_root, load_test_data, load_training_data

# ── configuration ─────────────────────────────────────────────────────────────
FEATURES = ["var_175", "var_190", "var_214", "var_341", "var_379", "var_482"]
FEATURE_INDICES = [176, 191, 215, 342, 380, 483]  # written to _vars.txt (1-indexed: V1=1)

MODEL_PARAMS = {
    "class_weight": "balanced",
    "max_depth": 5,
    "min_samples_leaf": 16,
    "n_estimators": 400,
    "random_state": 42,
}

TOP_K = 997  # OOF-optimal k (~20% contact rate)
SUBMISSION_PREFIX = "pozorski_florek_poltorak"
FEATURE_PENALTY = 200 * len(FEATURES)  # 1 200

root = find_project_root()
outputs = root / "outputs" / "feature_selection_alternative" / "final_submission"
outputs.mkdir(parents=True, exist_ok=True)

print(f"Project root : {root}")
print(f"Output dir   : {outputs}")
print(f"Features ({len(FEATURES)}): {FEATURES}")
print(f"TOP_K        : {TOP_K}")

## 2. Load data

In [ ]:
X_train, y_train = load_training_data(root / "data")
X_test = load_test_data(root / "data")

print(f"X_train : {X_train.shape}  |  positives: {y_train.sum()} ({y_train.mean():.1%})")
print(f"X_test  : {X_test.shape}")

# sanity check — all selected features must be present in both sets
missing_train = [f for f in FEATURES if f not in X_train.columns]
missing_test = [f for f in FEATURES if f not in X_test.columns]
if missing_train or missing_test:
    raise ValueError(f"Missing features — train: {missing_train}  test: {missing_test}")
print("All features present in both sets.")

## 3. Train on full X_train

The model is fitted on **all 5 000** training observations — no validation split here,
because k was already determined from OOF and the configuration was chosen via nested CV.

In [ ]:
clf = ExtraTreesClassifier(**MODEL_PARAMS)
clf.fit(X_train[FEATURES], y_train)

# in-sample check — in-sample upper bound on training data
train_scores = clf.predict_proba(X_train[FEATURES])[:, 1]
order = np.argsort(train_scores)[::-1]
ranked_y = y_train.values[order]
limit = min(TOP_K, len(ranked_y))
tp_train = int((ranked_y[:limit] == 1).sum())
fp_train = int((ranked_y[:limit] == 0).sum())
biz_train = 10 * tp_train - 5 * fp_train - FEATURE_PENALTY

print(f"In-sample (top-{limit}): TP={tp_train}  FP={fp_train}  score={biz_train:+.0f}")
print("(in-sample upper bound — expected higher than OOF)")

## 4. Score X_test and select top-k

In [ ]:
test_scores = clf.predict_proba(X_test[FEATURES])[:, 1]

test_df = pd.DataFrame({
    "sample_index": np.arange(1, len(X_test) + 1),  # 1-indexed: first row = 1
    "score": test_scores,
})
test_df = test_df.sort_values("score", ascending=False).reset_index(drop=True)
test_df["rank"] = test_df.index + 1

top_k_df = test_df.head(TOP_K).copy()

print(f"Scored {len(test_df)} test observations")
print(f"Selected top-{TOP_K} (contact rate: {TOP_K / len(test_df):.2%})")
print(
    f"\nScore — min: {top_k_df['score'].min():.4f}  "
    f"median: {top_k_df['score'].median():.4f}  "
    f"max: {top_k_df['score'].max():.4f}"
)
print()
top_k_df.head(10)

## 5. Save submission files

Required format:
- `_obs.txt` — one observation index per line (1-indexed rows of X_test)
- `_vars.txt` — one variable index per line (1-indexed, V1=1)

In [ ]:
obs_path = outputs / f"{SUBMISSION_PREFIX}_obs.txt"
vars_path = outputs / f"{SUBMISSION_PREFIX}_vars.txt"

obs_indices = top_k_df["sample_index"].to_numpy()  # already 1-indexed
np.savetxt(obs_path, obs_indices, fmt="%d")
np.savetxt(vars_path, FEATURE_INDICES, fmt="%d")

print(f"Saved {len(obs_indices)} observations -> {obs_path.relative_to(root)}")
print(f"Saved {len(FEATURE_INDICES)} variables  -> {vars_path.relative_to(root)}")
print()
print("First 10 observation indices:")
print(obs_indices[:10].tolist())
print()
print("Variable indices:")
print(FEATURE_INDICES)

## Summary

| | Value |
|---|---|
| Model | ExtraTrees (balanced, depth=5, leaf=16, trees=400) |
| Features | var_175, var_190, var_214, var_341, var_379, var_482 |
| Feature cost | 200 × 6 = 1 200 |
| Contact rate | 997 / 5 000 ≈ 19.94% |
| OOF business score | **5 425** (nested-CV OOF estimate — pooled 5×1 000 OOF) |
| In-sample bound | ~5 790 (in-sample upper bound) |

---

### How were the features and parameters chosen?

1. **Prescreening**: sparse GAM SPAM (generalised additive model with L1 penalty)
   fitted on the full X_train selected the top-6 variables by importance.

2. **Validation**: nested 5×5 CV (5 outer folds × 5 inner folds) — no observation
   was evaluated by a model that trained on it.

3. **k selection**: argmax sweep of business score on pooled OOF
   (5×1 000 = 5 000 disjoint predictions) → k=997.

4. **Feature stability**: `{var_175, var_379, var_482}` present in all 5 outer folds;
   `{var_190, var_214, var_341}` appear in ≥4 folds.

Full pipeline notebooks:
- `modeling_feature_selection_alternative_evaluation.ipynb` — nested CV + OOF scores
- `modeling_feature_selection_alternative_refit_ranking.ipynb` — refit + test ranking
- `score_comparison_oof_vs_test.ipynb` — score distribution and misclassification analysis